In [1]:
!pip install -Uq langchain-groq langchain dotenv langchain-community langchain-text-splitters langchain-huggingface langchain-core -qU

In [18]:
from langchain_community.document_loaders import WebBaseLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Narendra_Modi")
docs = loader.load()


text_spitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=10, separators=["\n\n", "\n", " ", ""])
chunks = text_spitter.split_documents(docs)

print(f"Total Chunks: {len(chunks)}")

Total Chunks: 668


#### Embeddings

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

chunk_embeddings = embeddings_model.embed_documents([chunk.page_content for chunk in chunks])

print(f"Total Chunk Embeddings: {len(chunk_embeddings)}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10605.46it/s]


Total Chunk Embeddings: 668


In [13]:
!pip install -qU langchain-chroma faiss-cpu

#### Vector stores

In [28]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="first_collection",
    embedding_function=embeddings_model,
    persist_directory="./chroma_langchain_db",
    
)

vector_store.add_documents(chunks)

results = vector_store.similarity_search("who is Narendra Damodardas Modi Spouse  ?", k=3)
for result in results:
    print(result.page_content)

Personal detailsBornNarendra Damodardas Modi (1950-09-17) 17 September 1950 (age 76)Vadnagar, Bombay State, IndiaPartyBharatiya Janata Party(since 1985)Spouse
Jashodaben Modi
​ ​(m. 1968; sep. 1971)​[2]EducationUniversity of Delhi (BA)Gujarat University (MA)AwardsList of awards and honoursSignatureWebsitePersonalPM India official
Personal detailsBornNarendra Damodardas Modi (1950-09-17) 17 September 1950 (age 76)Vadnagar, Bombay State, IndiaPartyBharatiya Janata Party(since 1985)Spouse
Jashodaben Modi
​ ​(m. 1968; sep. 1971)​[2]EducationUniversity of Delhi (BA)Gujarat University (MA)AwardsList of awards and honoursSignatureWebsitePersonalPM India official
Personal detailsBornNarendra Damodardas Modi (1950-09-17) 17 September 1950 (age 76)Vadnagar, Bombay State, IndiaPartyBharatiya Janata Party(since 1985)Spouse
Jashodaben Modi
​ ​(m. 1968; sep. 1971)​[2]EducationUniversity of Delhi (BA)Gujarat University (MA)AwardsList of awards and honoursSignatureWebsitePersonalPM India official


In [35]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings_model)

vector_store.index.ntotal

vector_store.save_local("./faiss_langchain_db")

In [38]:
vector_store = FAISS.load_local(
    "./faiss_langchain_db",
    embeddings_model,
    allow_dangerous_deserialization=True,
)


results = vector_store.similarity_search("who is Narendra Damodardas Modi Spouse  ?", k=3)
for result in results:
    print(result.page_content)


Personal detailsBornNarendra Damodardas Modi (1950-09-17) 17 September 1950 (age 76)Vadnagar, Bombay State, IndiaPartyBharatiya Janata Party(since 1985)Spouse
Jashodaben Modi
​ ​(m. 1968; sep. 1971)​[2]EducationUniversity of Delhi (BA)Gujarat University (MA)AwardsList of awards and honoursSignatureWebsitePersonalPM India official
In a custom traditional to Narendra Modi's caste, his family arranged a betrothal to Jashodaben Chimanlal Modi, leading to their marriage when she was 17 and he was 18.[40][41] The marriage was never consummated, and Modi soon abandoned his wife,[42][43] and left home. The couple never divorced but the marriage was not in his public pronouncements for many decades.[41] In April 2014, shortly before the national election in which he gained power, Modi publicly affirmed he was married and that
↑ Mahurkar, Uday (10 April 2014). "Revealed: Why Narendra Modi walked out of his marriage with Jashodaben". India Today. Retrieved 29 December 2025.
↑ "Narendra Modi Fast 

#### Retervier

In [41]:
!pip install wikipedia -qU

In [44]:
from langchain_community.retrievers import WikipediaRetriever
retriever = WikipediaRetriever()

docs = retriever.invoke("who is Narendra Damodardas Modi Spouse  ?")

print(docs[0].page_content)

Narendra Damodardas Modi (born 17 September 1950) is an Indian politician who has served as the  prime minister of India since 26 May 2014. Modi was the chief minister of Gujarat from 2001 to 2014 and is the Member of Parliament (MP) for Varanasi. He is a member of the Bharatiya Janata Party (BJP) and of the Rashtriya Swayamsevak Sangh (RSS), a right-wing Hindutva paramilitary volunteer organisation. He is India's third-longest-serving prime minister, and the longest-serving prime minister outside the Indian National Congress.
Modi was born and raised in Vadnagar, where he completed his secondary education. He was introduced to the RSS at the age of eight, becoming a full-time worker for the organisation in Gujarat in 1971. Assigned by the RSS to the BJP in 1985, he rose through the party hierarchy and became general secretary in 1998. In 2001, Modi was appointed chief minister of Gujarat and elected to the legislative assembly soon after. His administration is considered complicit in 